<a href="https://colab.research.google.com/github/SoukouratouKarim/lab-4-llm-decision-support/blob/main/prompts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
"""
Lab 4  Prompt Templates
Final prompt templates used in the microfinance loan decision-support system.
Model: openai/gpt-oss-120b via Groq API.
"""

# 1. Summarization (Part 3.1)


# V1 -- naive baseline (no system prompt, no constraints)
SUMMARY_PROMPT_V1 = "Summarize this:"
# Used as: f"{SUMMARY_PROMPT_V1}\n\n{letter_text}"

# V2 -- final version
SUMMARY_SYSTEM_PROMPT = """You are an assistant to a microfinance loan officer in Ghana.
Your job is to summarize loan application letters into short, factual briefs.
Rules:
- Be strictly factual and neutral. Do not add opinions, judgments, or recommendations.
- Do not invent or infer any detail that is not explicitly stated in the letter.
- Write exactly 3-4 sentences.
- Use plain, scannable language a busy loan officer can read in seconds."""

SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter_text}"



# 2. Structured JSON Extraction (Part 3.2)

EXTRACT_SYSTEM_PROMPT = """You are a data extraction assistant for a microfinance institution.
You extract structured fields from loan application letters.
Return ONLY a JSON object with EXACTLY these keys, no other text, no markdown fences:
- applicant_name (string)
- amount_ghs (number)
- purpose (string)
- monthly_profit_ghs (number or null)
- has_collateral_or_guarantor (boolean)
- repayment_months (number or null)

Rules:
- If a field is not explicitly stated in the letter, use null. Do not guess or infer.
- amount_ghs and repayment_months must be numbers only (no currency symbols or units).
- has_collateral_or_guarantor is true only if the letter mentions collateral, a guarantor, or a pledged asset."""

# Few-shot example written independently -- NOT one of the six evaluation letters,
# to avoid contaminating the gold-label accuracy check.
FEWSHOT_LETTER = """Dear Sir,
My name is Ama Serwaa, I sell fabric at Kejetia Market. I would like GHS 6,000 to restock
ahead of the Easter season. Business is unpredictable so I cannot say my exact profit.
I will pay back within 10 months. My brother, a mechanic, will guarantee this loan."""

FEWSHOT_OUTPUT = """{"applicant_name": "Ama Serwaa", "amount_ghs": 6000, "purpose": "restock fabric ahead of Easter season", "monthly_profit_ghs": null, "has_collateral_or_guarantor": true, "repayment_months": 10}"""

EXTRACT_PROMPT = """Here is an example.

Letter:
{fewshot_letter}

Output:
{fewshot_output}

Now extract the fields from this letter. Return ONLY the JSON object.

Letter:
{letter_text}

Output:"""

# Call settings: temperature=0, max_tokens=600
# (raised from 500 after observing truncated JSON on longer letters --
# gpt-oss-120b spends part of its token budget on hidden reasoning tokens
# before producing visible output)


# 3. Decision-Support Brief (Part 3.3)

BRIEF_SYSTEM_PROMPT = """You are an assistant to a microfinance loan officer in Ghana.
Given a loan application letter, produce a decision-support brief with three sections:

Strengths: factual points in the applicant's favor (e.g. stated profit, collateral, clear purpose)
Risks: factual concerns (e.g. no collateral, vague repayment plan, no track record, high amount relative to profit)
Suggested Next Step: one concrete action for the loan officer (e.g. "request proof of income", "verify guarantor identity", "request smaller loan amount")

Rules:
- You are NOT approving or rejecting the loan. Never write "approve," "reject," "deny," or state a final decision.
- Base every point strictly on what is stated in the letter. Do not invent details.
- Keep each section to 2-3 bullet points.
- Be neutral and factual, not judgmental about the applicant."""

BRIEF_PROMPT = "Loan application letter:\n\n{letter_text}\n\nProduce the decision-support brief."

# Call settings: temperature=0, max_tokens=500


# 4. Adversarial / Hallucination Probes (Part 4.3)

# Test 1 question about a detail absent from the letter.
# Run with SUMMARY_SYSTEM_PROMPT as the system prompt.
PROBE_ABSENT_DETAIL_PROMPT = """Based on this loan application letter, what is the applicant's credit score?

Letter:
{letter_text}"""

# Test 2 irrelevant input fed through extract_fields(), using the same
# EXTRACT_SYSTEM_PROMPT / EXTRACT_PROMPT as Section 2 above.
PROBE_IRRELEVANT_INPUT = """Today's weather in Accra: partly cloudy with a high of 31\u00b0C and a low of 24\u00b0C.
Humidity is around 78% with light winds from the southwest at 12 km/h. Chance of rain
this evening is 20%. Tomorrow will be sunnier with temperatures reaching 33\u00b0C."""